# Baseline 01 — Classical ML Defect Classification

This notebook establishes the first interpretable baseline for FactoryGuard AI.

Scope for this baseline:
- dataset inspection and EDA
- train/dev/test split
- random-guess baseline
- logistic regression baseline (flattened pixel features)
- evaluation via Accuracy, Macro F1, Confusion Matrix
- misclassification analysis and hypotheses for next experiments

> Deep learning is intentionally out of scope for Baseline 01.

In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set(style='whitegrid')

## 1) Load and inspect dataset

Expected structure under `data/raw/neu_surface_defect/` with one folder per class.

In [ ]:
DATA_DIR = Path('../data/raw/neu_surface_defect')

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATA_DIR.resolve()}
"
        "Download/extract NEU dataset into data/raw/neu_surface_defect before running this notebook."
    )

image_paths = sorted([p for p in DATA_DIR.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}])
if not image_paths:
    raise RuntimeError(f'No image files found under {DATA_DIR.resolve()}')

records = []
for p in image_paths:
    label = p.parent.name
    with Image.open(p) as img:
        width, height = img.size
        mode = img.mode
    records.append({
        'path': str(p),
        'label': label,
        'width': width,
        'height': height,
        'mode': mode,
    })

df = pd.DataFrame(records)
print('Total images:', len(df))
df.head()

## 2) Check image dimensions and formats

In [ ]:
print('Unique image modes:', sorted(df['mode'].unique().tolist()))
print('
Top image size frequencies:')
print(df.groupby(['width', 'height']).size().sort_values(ascending=False).head(10))

## 3) Class counts and class distribution

In [ ]:
class_counts = df['label'].value_counts().sort_index()
print(class_counts)

plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.xticks(rotation=30, ha='right')
plt.title('Samples per class')
plt.xlabel('Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4) Visualize representative images

In [ ]:
labels = sorted(df['label'].unique())
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for ax, lbl in zip(axes, labels):
    sample_path = df[df['label'] == lbl]['path'].iloc[0]
    img = Image.open(sample_path)
    ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
    ax.set_title(lbl)
    ax.axis('off')

for ax in axes[len(labels):]:
    ax.axis('off')

plt.suptitle('Representative images by class', y=1.02)
plt.tight_layout()
plt.show()

## 5) Define problem and metrics

- Multi-class defect classification
- Metrics: Accuracy, Macro F1, Confusion Matrix

## 6) Prepare flattened image features and split train/dev/test

In [ ]:
# Convert images to a common size and grayscale for classical baseline
IMG_SIZE = (64, 64)

X = []
y = []
for _, row in df.iterrows():
    with Image.open(row['path']) as img:
        img = img.convert('L').resize(IMG_SIZE)
        arr = np.array(img, dtype=np.float32).reshape(-1) / 255.0
    X.append(arr)
    y.append(row['label'])

X = np.array(X)
y = np.array(y)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_

# 70/15/15 split using stratification
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.30, random_state=SEED, stratify=y_encoded
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print('Train size:', len(y_train))
print('Dev size:  ', len(y_dev))
print('Test size: ', len(y_test))

## 7) Random-guess baseline

In [ ]:
dummy = DummyClassifier(strategy='stratified', random_state=SEED)
dummy.fit(X_train, y_train)

y_dev_dummy = dummy.predict(X_dev)

dummy_acc = accuracy_score(y_dev, y_dev_dummy)
dummy_f1 = f1_score(y_dev, y_dev_dummy, average='macro')
print(f'Random baseline (dev) Accuracy: {dummy_acc:.4f}')
print(f'Random baseline (dev) Macro F1: {dummy_f1:.4f}')

## 8) Logistic Regression baseline

In [ ]:
logreg = make_pipeline(
    StandardScaler(with_mean=False),
    LogisticRegression(
        max_iter=2000,
        multi_class='multinomial',
        solver='lbfgs',
        random_state=SEED,
        n_jobs=None,
    )
)

logreg.fit(X_train, y_train)

y_dev_pred = logreg.predict(X_dev)

dev_acc = accuracy_score(y_dev, y_dev_pred)
dev_f1 = f1_score(y_dev, y_dev_pred, average='macro')
print(f'LogReg (dev) Accuracy: {dev_acc:.4f}')
print(f'LogReg (dev) Macro F1: {dev_f1:.4f}')

print('
Classification report (dev):')
print(classification_report(y_dev, y_dev_pred, target_names=class_names, zero_division=0))

## 9) Confusion Matrix

In [ ]:
cm = confusion_matrix(y_dev, y_dev_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Logistic Regression Confusion Matrix (Dev)')
plt.tight_layout()
plt.show()

## 10) Visualize misclassified examples

In [ ]:
mis_idx = np.where(y_dev != y_dev_pred)[0]
print('Number of misclassified dev samples:', len(mis_idx))

n_show = min(12, len(mis_idx))
if n_show == 0:
    print('No misclassifications to visualize.')
else:
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    axes = axes.flatten()

    for ax_i, idx in enumerate(mis_idx[:n_show]):
        ax = axes[ax_i]
        img = X_dev[idx].reshape(IMG_SIZE)
        true_lbl = class_names[y_dev[idx]]
        pred_lbl = class_names[y_dev_pred[idx]]
        ax.imshow(img, cmap='gray')
        ax.set_title(f'True: {true_lbl}
Pred: {pred_lbl}')
        ax.axis('off')

    for ax in axes[n_show:]:
        ax.axis('off')

    plt.suptitle('Misclassified Dev Examples', y=1.02)
    plt.tight_layout()
    plt.show()

## 11) Optional final test-set evaluation

Run this only after selecting the baseline configuration from dev results.

In [ ]:
# Uncomment after model selection is complete:
# y_test_pred = logreg.predict(X_test)
# print('LogReg (test) Accuracy:', accuracy_score(y_test, y_test_pred))
# print('LogReg (test) Macro F1:', f1_score(y_test, y_test_pred, average='macro'))

## 12) Observations and hypotheses

Use this section to record:
- What was changed and why
- Current metrics
- Common failure patterns from confusion matrix/misclassifications
- Hypotheses for next experiments (e.g., HOG features, regularization, data quality checks)